# Iceberg Basics — 07: Partition Evolution


Partition evolution is Iceberg's most unique feature — you can change
the partitioning strategy without rewriting any existing data.
Old data stays in its original layout; new data uses the new scheme.

Topics: changing partition transforms, mixed-layout reads, migration patterns.


In [ ]:
import os, time, pathlib, shutil, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/iceberg_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('iceberg-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark.sql("CREATE DATABASE IF NOT EXISTS local.icedb")
print(f"Spark {spark.version} | Iceberg catalog ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

## Step 1 — Start with Monthly Partitioning

Create table with months(order_date).

In [ ]:

spark.sql("DROP TABLE IF EXISTS local.icedb.evolve_orders")
(df.filter(F.col("order_date") < datetime.date(2024,4,1))
   .writeTo("local.icedb.evolve_orders")
   .partitionedBy(F.partitioning.months("order_date"))
   .using("iceberg")
   .createOrReplace())
# Write Q1 2024 data
q1 = spark.table("local.icedb.evolve_orders")
print(f"Q1 data written (monthly partitioned): {spark.table('local.icedb.evolve_orders').count():,} rows")
spark.sql("SELECT partition, record_count FROM local.icedb.evolve_orders.partitions ORDER BY partition LIMIT 4").show()


## Step 2 — Evolve to Daily Partitioning

Replace months with days — old data stays monthly, new data is daily.

In [ ]:

spark.sql("""
    ALTER TABLE local.icedb.evolve_orders
    REPLACE PARTITION FIELD months(order_date)
    WITH days(order_date)
""")
print("Partition evolved: months → days")
print("Old Q1 files STAY in monthly layout — no rewrite!")
print()

# Write Q2 2024 data — goes to daily partitions
q2 = df.filter(
    (F.col("order_date") >= datetime.date(2024,4,1)) &
    (F.col("order_date") < datetime.date(2024,7,1))
)
q2.writeTo("local.icedb.evolve_orders").append()
print(f"Q2 data written (daily partitioned): {q2.count():,} rows")
print()

# Iceberg reads BOTH layouts seamlessly
total = spark.table("local.icedb.evolve_orders").count()
q1_check = spark.table("local.icedb.evolve_orders") \
                .filter(F.col("order_date") < datetime.date(2024,4,1)).count()
q2_check = spark.table("local.icedb.evolve_orders") \
                .filter(F.col("order_date") >= datetime.date(2024,4,1)).count()
print(f"Total readable: {total:,} rows  (Q1={q1_check:,} monthly + Q2={q2_check:,} daily)")
print("Iceberg reads both partition layouts transparently ✅")


## Step 3 — Partition History

Inspect partition spec history.

In [ ]:

import json as pyjson, pathlib
warehouse = "/workspace/data/warehouse/iceberg"
meta_dir = pathlib.Path(f"{warehouse}/icedb/evolve_orders/metadata")
meta_files = sorted(meta_dir.glob("*.json"))
if meta_files:
    meta = pyjson.loads(meta_files[-1].read_text())
    specs = meta.get("partition-specs", [])
    print("Partition spec history:")
    for spec in specs:
        fields = [f"{f['transform']}({f['source-id']})" for f in spec.get("fields", [])]
        print(f"  spec-id={spec['spec-id']}: {fields}")
    print()
    print(f"current-spec-id: {meta.get('current-spec-id')}")
    print()
    print("Each data file records which spec-id was used to write it.")
    print("Reader uses the correct spec per file → transparent mixed-layout reads.")


## Summary



In [ ]:

print("""
-- Change partition transform (no data rewrite)
ALTER TABLE local.db.table
REPLACE PARTITION FIELD months(ts) WITH days(ts)

-- Add partition column
ALTER TABLE local.db.table ADD PARTITION FIELD category

-- Remove partition column
ALTER TABLE local.db.table DROP PARTITION FIELD category

Benefits:
  - Old data files stay in original layout (no migration cost)
  - New data uses new partition scheme immediately
  - Iceberg reads both layouts transparently via spec-id per file
  - Impossible in Hive/plain Parquet without full table rewrite
""")
